## Multimodal Interactions

**Topic:** CLIP Retrieval + BLIP Captioning + Simple Agentic Pipeline

This notebook builds a small multimodal system that uses:
- **CLIP** as an image retrieval system over a database of 100 images
- **BLIP** as a captioning tool for grounding and verification
- agentic loop that retrieves images, captions selected results, and produces a final answer


In [12]:
from pathlib import Path
import json

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from transformers import (
    CLIPModel,
    CLIPProcessor,
    BlipProcessor,
    BlipForConditionalGeneration,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# image data set up, create image index
image_dir = Path("assignment4_images")
image_path = sorted(image_dir.glob("*.jpg"))
image_ids = [path.name for path in image_path]


# helper dictionary for future direct image questioning
image_id_to_path = {path.name: path for path in image_path}

Using device: cuda


### 2. CLIP Retrieval Setup

In [2]:
clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_model.eval() ## put in inference mode since we are not training

print("Loaded CLIP model:", clip_model_name, device)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded CLIP model: openai/clip-vit-base-patch32 cuda


### 3. Build Image Embeddings

In [13]:
batch_size = 16

# Re-check image list in case runtime/kernel path changed
if len(image_path) == 0:
    image_path = sorted(image_dir.glob("*.jpg"))

if len(image_path) == 0:
    image_embeddings = None
    print("No .jpg images found in assignment4_images. Add images, then rerun this cell.")
else:
    image_ids = [path.name for path in image_path]
    image_id_to_path = {path.name: path for path in image_path}

    all_image_features = []
    for start_idx in range(0, len(image_path), batch_size):
        batch_paths = image_path[start_idx:start_idx + batch_size]
        batch_images = []
        for path in batch_paths:
            with Image.open(path) as img:
                batch_images.append(img.convert("RGB"))

        inputs = clip_processor(images=batch_images, return_tensors="pt").to(device)

        with torch.no_grad():
            features = clip_model.get_image_features(**inputs)
            # Extract tensor if an object is returned
            if not isinstance(features, torch.Tensor):
                features = features.pooler_output if hasattr(features, 'pooler_output') else features[0]

            features = features / features.norm(dim=-1, keepdim=True)

        all_image_features.append(features.cpu())

    image_embeddings = torch.cat(all_image_features, dim=0)
    print("Built image embeddings")
    print("image_embeddings shape:", tuple(image_embeddings.shape))
    print("number of image_ids:", len(image_ids))


Built image embeddings
image_embeddings shape: (100, 512)
number of image_ids: 100


### 4. Text Query Retrieval (Top-k)

In [18]:
query = "a dog playing in water"
top_k = 5

assert image_embeddings is not None and len(image_ids) > 0, "Run Cell 6 first to build image embeddings."

text_inputs = clip_processor(
    text=[query],
    return_tensors="pt",
    padding=True,
).to(device)

with torch.no_grad():
    # get_text_features directly returns the pooled text_embeds for the sentence
    text_features = clip_model.get_text_features(**text_inputs)

    # Ensure we only have a 2D tensor (batch_size, embedding_dim)
    if text_features.ndim == 3:
        # Fallback to pooler output if we accidentally got token-level hidden states
        text_features = text_features[:, 0, :]

    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

similarities = (text_features.cpu() @ image_embeddings.T).squeeze()
k = min(top_k, len(image_ids))
top_scores, top_indices = torch.topk(similarities, k=k)

retrieved_items = []
print(f"Query: {query}")
print(f"Top {k} retrieved images:")

# Safely iterate over the 1D top_k arrays
for rank, (score, idx) in enumerate(zip(top_scores.view(-1).tolist(), top_indices.view(-1).tolist()), start=1):
    image_name = image_ids[idx]
    retrieved_items.append((image_name, float(score)))
    print(f"{rank}. {image_name} | score={score:.4f}")

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'ndim'

### 5. BLIP Captioning for Retrieved Images

In [ ]:
# Load BLIP captioning model once
if "blip_model" not in globals() or "blip_processor" not in globals():
    blip_model_name = "Salesforce/blip-image-captioning-base"
    blip_processor = BlipProcessor.from_pretrained(blip_model_name)
    blip_model = BlipForConditionalGeneration.from_pretrained(blip_model_name).to(device)
    blip_model.eval()
    print("Loaded BLIP model:", blip_model_name, device)

assert "retrieved_items" in globals() and len(retrieved_items) > 0, "Run Cell 8 first to retrieve top images."

captioned_items = []
print(f"Query: {query}")
print(f"Top {len(retrieved_items)} images with BLIP captions:")
for rank, (image_name, score) in enumerate(retrieved_items, start=1):
    image_file = image_id_to_path[image_name]

    with Image.open(image_file).convert("RGB") as img:
        blip_inputs = blip_processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            generated = blip_model.generate(**blip_inputs, max_new_tokens=30)
        caption = blip_processor.decode(generated[0], skip_special_tokens=True)

    captioned_items.append({
        "image": image_name,
        "score": float(score),
        "caption": caption,
    })

    print(f"{rank}. {image_name} | score={score:.4f}")
    print(f"   caption: {caption}")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loaded BLIP model: Salesforce/blip-image-captioning-base cpu
Build image embeddings first by running Cell 6.


### 6. Simple Agentic Pipeline Output

In [ ]:
assert "captioned_items" in globals() and len(captioned_items) > 0, "Run Cell 10 first to generate captions."

top_context = captioned_items[:3]
best_match = top_context[0]

final_answer = (
    f"For the query '{query}', the best match is {best_match['image']} "
    f"with caption: {best_match['caption']}"
)

agent_output = {
    "query": query,
    "top_context": top_context,
    "final_answer": final_answer,
}

print("Agent summary:")
print(final_answer)
print("\nStructured output:")
print(json.dumps(agent_output, indent=2))

### 7. Visualize Retrieved Results

In [ ]:
assert "top_context" in globals() and len(top_context) > 0, "Run Cell 12 first."

num_items = len(top_context)
fig, axes = plt.subplots(1, num_items, figsize=(5 * num_items, 5))

if num_items == 1:
    axes = [axes]

for i, item in enumerate(top_context):
    image_name = item["image"]
    score = item["score"]
    caption = item["caption"]

    with Image.open(image_id_to_path[image_name]).convert("RGB") as img:
        axes[i].imshow(img)

    axes[i].axis("off")
    axes[i].set_title(f"#{i + 1} {image_name}\nscore={score:.4f}", fontsize=10)
    axes[i].set_xlabel(caption, fontsize=9)

plt.tight_layout()
plt.show()